# 02 - Data Cleaning

This notebook prepares a clean analysis dataset from the Global Findex 2025 CSV.

The goal is to load only the selected columns, rename coded indicators into business-friendly names, check missing values, validate row uniqueness, and decide what cleaning rules are needed before SQL and Power BI analysis.

This notebook does not final conclusions. It prepares dataset for analysis.

## Import Libraries

In [1]:
from pathlib import Path
import pandas as pd

## Set File Path

In [2]:
DATA_PATH = Path(r"D:/financial-inclusion-gap-analysis/data/raw/GlobalFindexDatabase2025.csv")

DATA_PATH.exists()

True

## Define Selected Columns

In [3]:
context_columns = [
    "countrynewwb",
    "codewb",
    "year",
    "pop_adult",
    "regionwb24_hi",
    "incomegroupwb24",
    "group",
    "group2"
]

indicator_columns = [
    "account_t_d",
    "fiaccount_t_d",
    "mobileaccount_t_d",
    "dig_acc",
    "merchant_pay",
    "g20_made",
    "g20_received",
    "inactive_t_d"
]

selected_columns = context_columns + indicator_columns

selected_columns

['countrynewwb',
 'codewb',
 'year',
 'pop_adult',
 'regionwb24_hi',
 'incomegroupwb24',
 'group',
 'group2',
 'account_t_d',
 'fiaccount_t_d',
 'mobileaccount_t_d',
 'dig_acc',
 'merchant_pay',
 'g20_made',
 'g20_received',
 'inactive_t_d']

## Load Selected Columns Only

In [4]:
df = pd.read_csv(
    DATA_PATH,
    usecols=selected_columns,
    low_memory=False
)

df.head()

,countrynewwb,codewb,year,pop_adult,regionwb24_hi,incomegroupwb24,group,group2,account_t_d,fiaccount_t_d,mobileaccount_t_d,dig_acc,inactive_t_d,g20_made,g20_received,merchant_pay
0,Afghanistan,AFG,2011,14575546.0,South Asia (excluding high income),Low income,all,all,0.090050,0.090050,NaN,NaN,NaN,NaN,NaN,NaN
1,Albania,ALB,2011,2281010.0,Europe & Central Asia (excluding high income),Upper middle income,all,all,0.282681,0.282681,NaN,NaN,NaN,NaN,NaN,NaN
2,Algeria,DZA,2011,26251587.0,Middle East & North Africa (excluding high inc...,Lower middle income,all,all,0.332861,0.332861,NaN,NaN,NaN,NaN,NaN,NaN
3,Angola,AGO,2011,12779501.0,Sub-Saharan Africa (excluding high income),Lower middle income,all,all,0.392035,0.392035,NaN,NaN,NaN,NaN,NaN,NaN
4,Argentina,ARG,2011,30685516.0,Latin America & Caribbean (excluding high income),Upper middle income,all,all,0.331302,0.331302,NaN,NaN,NaN,NaN,NaN,NaN


## Basic Shape Check

In [5]:
df.shape

(8577, 16)

## Inspect Structure

Check column names, data types, and not-null counts before cleaning

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8577 entries, 0 to 8576
Data columns (total 16 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   countrynewwb       8577 non-null   object 
 1   codewb             8577 non-null   object 
 2   year               8577 non-null   int64  
 3   pop_adult          7893 non-null   float64
 4   regionwb24_hi      7893 non-null   object 
 5   incomegroupwb24    7893 non-null   object 
 6   group              8577 non-null   object 
 7   group2             8577 non-null   object 
 8   account_t_d        8487 non-null   float64
 9   fiaccount_t_d      8395 non-null   float64
 10  mobileaccount_t_d  2527 non-null   float64
 11  dig_acc            1375 non-null   float64
 12  inactive_t_d       869 non-null    float64
 13  g20_made           5780 non-null   float64
 14  g20_received       5778 non-null   float64
 15  merchant_pay       1754 non-null   float64
dtypes: float64(9), int64(1),

## Rename Columns

In [7]:
rename_map = {
    "countrynewwb": "country",
    "codewb": "country_code",
    "pop_adult": "adult_population",
    "regionwb24_hi": "region",
    "incomegroupwb24": "income_group",
    "account_t_d": "account_ownership",
    "fiaccount_t_d": "financial_institution_account",
    "mobileaccount_t_d": "mobile_money_account",
    "dig_acc": "digital_access",
    "merchant_pay": "digital_merchant_payment",
    "g20_made": "digital_payment_made",
    "g20_received": "digital_payment_received",
    "inactive_t_d": "inactive_account"
}

df = df.rename(columns=rename_map)

df.head()

,country,country_code,year,adult_population,region,income_group,group,group2,account_ownership,financial_institution_account,mobile_money_account,digital_access,inactive_account,digital_payment_made,digital_payment_received,digital_merchant_payment
0,Afghanistan,AFG,2011,14575546.0,South Asia (excluding high income),Low income,all,all,0.090050,0.090050,NaN,NaN,NaN,NaN,NaN,NaN
1,Albania,ALB,2011,2281010.0,Europe & Central Asia (excluding high income),Upper middle income,all,all,0.282681,0.282681,NaN,NaN,NaN,NaN,NaN,NaN
2,Algeria,DZA,2011,26251587.0,Middle East & North Africa (excluding high inc...,Lower middle income,all,all,0.332861,0.332861,NaN,NaN,NaN,NaN,NaN,NaN
3,Angola,AGO,2011,12779501.0,Sub-Saharan Africa (excluding high income),Lower middle income,all,all,0.392035,0.392035,NaN,NaN,NaN,NaN,NaN,NaN
4,Argentina,ARG,2011,30685516.0,Latin America & Caribbean (excluding high income),Upper middle income,all,all,0.331302,0.331302,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
df.columns.tolist()

['country',
 'country_code',
 'year',
 'adult_population',
 'region',
 'income_group',
 'group',
 'group2',
 'account_ownership',
 'financial_institution_account',
 'mobile_money_account',
 'digital_access',
 'inactive_account',
 'digital_payment_made',
 'digital_payment_received',
 'digital_merchant_payment']

## Missing-Value Assessment

Missing values are measured before cleaning. Some indicators may be unavailable for particular years or population groups, so missing values should not automatically be treated as errors.


In [9]:
missing_summary = pd.DataFrame({
    "missing_count": df.isna().sum(),
    "missing_percentage": (df.isna().mean() * 100).round(2)
}).sort_values("missing_percentage", ascending=False)

missing_summary

,missing_count,missing_percentage
inactive_account,7708,89.87
digital_access,7202,83.97
digital_merchant_payment,6823,79.55
mobile_money_account,6050,70.54
digital_payment_received,2799,32.63
digital_payment_made,2797,32.61
adult_population,684,7.97
region,684,7.97
income_group,684,7.97
financial_institution_account,182,2.12


In [10]:
indicator_columns_clean=[
    "account_ownership",
    "financial_institution_account",
    "mobile_money_account",
    "digital_access",
    "digital_merchant_payment",
    "digital_payment_made",
    "digital_payment_received",
    "inactive_account"
]

missing_by_year = (df.groupby("year")[indicator_columns_clean].apply(
    lambda group: group.isna().mean().mul(100).round(2))
                  )
missing_by_year

,account_ownership,financial_institution_account,mobile_money_account,digital_access,digital_merchant_payment,digital_payment_made,digital_payment_received,inactive_account
year,,,,,,,,
2011,0.00,0.00,100.00,100.00,100.00,100.00,100.00,100.00
2014,3.56,5.34,85.41,100.00,100.00,20.17,20.76,86.83
2017,0.58,2.31,71.74,100.00,100.00,8.07,7.61,86.97
2021,1.35,1.35,56.98,100.00,57.72,4.86,4.86,86.31
2022,0.00,11.36,40.91,100.00,56.82,11.36,11.36,85.23
2024,0.00,0.61,47.07,30.63,46.97,35.77,35.77,90.31


## Validate the Unit of Analysis

Each row is expected to represent one country-year-population-segment observation. The proposed key is country code, year, group, and subgroup

In [11]:
key_column = ["country_code", "year", "group", "group2"]

df[key_column].isna().sum()

country_code    0
year            0
group           0
group2          0
dtype: int64

In [12]:
duplicate_count = df.duplicated(
    subset=key_column,
    keep=False
).sum()

duplicate_count

The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.


np.int64(0)

In [13]:
duplicate_rows = (
    df[df.duplicated(subset=key_column, keep=False)]
    .sort_values(key_column)
)

duplicate_rows[key_column].head(20)

,country_code,year,group,group2


## Validate Indicator Ranges

Financial inclusion indicator are stored as decimal proportions. Valid non-missing values should fall between 0 and 1.

In [14]:
indicator_summary = df[indicator_columns_clean].describe().T[
    ["count", "mean", "min", "max"]
]

indicator_summary

,count,mean,min,max
account_ownership,8487.0,0.608645,0.004049,1.000000
financial_institution_account,8395.0,0.577825,0.004049,1.000000
mobile_money_account,2527.0,0.284001,0.000000,0.939388
digital_access,1375.0,0.463432,0.028120,0.984625
digital_merchant_payment,1754.0,0.327419,0.000000,0.972780
digital_payment_made,5780.0,0.468842,0.006904,1.000000
digital_payment_received,5778.0,0.400943,0.004907,0.988207
inactive_account,869.0,0.078093,0.000000,0.409191


In [15]:
invalid_range_counts = (
    ((df[indicator_columns_clean] < 0) | 
     (df[indicator_columns_clean] > 1)
    ).sum()
)

invalid_range_counts

account_ownership                0
financial_institution_account    0
mobile_money_account             0
digital_access                   0
digital_merchant_payment         0
digital_payment_made             0
digital_payment_received         0
inactive_account                 0
dtype: int64

## Validate Categories and Survey Years

Review categorical labels and survey-year coverage to detect inconsistent spelling, unexpected categories, or special survey periods.

In [16]:
categorical_columns = [
    "year",
    "region",
    "income_group",
    "group",
    "group2"
]

for column in categorical_columns:
    print(f"\n--- {column} ---")
    print(df[column].value_counts(dropna=False))


--- year ---
year
2024    1982
2017    1734
2014    1686
2011    1516
2021    1483
2022     176
Name: count, dtype: int64

--- region ---
region
High income                                           2643
Sub-Saharan Africa (excluding high income)            1883
Europe & Central Asia (excluding high income)         1038
Latin America & Caribbean (excluding high income)      930
NaN                                                    684
Middle East & North Africa (excluding high income)     537
East Asia & Pacific (excluding high income)            521
South Asia (excluding high income)                     341
Name: count, dtype: int64

--- income_group ---
income_group
High income            2643
Lower middle income    2211
Upper middle income    2089
Low income              950
NaN                     684
Name: count, dtype: int64

--- group ---
group
gender        1510
age_cat       1510
education     1508
laborforce    1490
income        1484
all            772
urbanicity     303
N

In [17]:
df.loc[
    df["year"] == 2022,
    ["country", "country_code", "year", "group", "group2"]
].head(20)

,country,country_code,year,group,group2
439,Azerbaijan,AZE,2022,all,all
445,Botswana,BWA,2022,all,all
452,Chad,TCD,2022,all,all
456,Comoros,COM,2022,all,all
457,"Congo, Dem. Rep.",COD,2022,all,all
470,Eswatini,SWZ,2022,all,all
471,Ethiopia,ETH,2022,all,all
475,"Gambia, The",GMB,2022,all,all
480,Guatemala,GTM,2022,all,all
504,Lesotho,LSO,2022,all,all


In [18]:
df.loc[df["year"] == 2022, "country_code"].nunique()

16

## Day 4 Progress Summary

- Loaded 16 selected columns from the raw dataset.
- Renamed coded columns into readable analytical names.
- Profiled missing values overall and by survey year.
- Confirmed that the proposed composite key has no missing values or duplicates.
- Confirmed that all non-missing indicator values fall between 0 and 1.
- Reviewed categorical values and identified 2022 as an off-cycle survey year covering 16 countries.
- No rows were removed and no missing values were filled at this stage.

### Next Step

Document the final cleaning rules, prepare the analysis-ready dataset, and validate the processed output before export.

## Identify Country and Aggregrate Rows

Some rows represent individual countries, while others represent global, regional, or income-group benchmarks. These must be distinguished to avoid mixing different entity types during analysis.

In [19]:
context_missing_rows = df[
    df["region"].isna() | df["income_group"].isna()
]

context_missing_rows[ 
    ["country", "country_code", "region", "income_group"]
].drop_duplicates()

,country,country_code,region,income_group
7893,world,WLD,NaN,NaN
7950,Developing economies,LMY,NaN,NaN
8007,East Asia & Pacific (excluding high income),EAP,NaN,NaN
8008,Europe & Central Asia (excluding high income),ECA,NaN,NaN
8009,Middle East & North Africa (excluding high inc...,MNA,NaN,NaN
8010,Sub-Saharan Africa (excluding high income),SSA,NaN,NaN
8195,Latin America & Caribbean (excluding high income),LAC,NaN,NaN
8196,South Asia,SAS,NaN,NaN
8349,High income,HIC,NaN,NaN
8350,Low income,LIC,NaN,NaN


## Create `entity_type`

In [20]:
aggregate_codes = context_missing_rows["country_code"].unique()

df["entity_type"] = "Country"

df.loc[
    df["country_code"].isin(aggregate_codes),
    "entity_type"
] = "Aggregate"

In [21]:
df["entity_type"].value_counts()

entity_type
Country      7893
Aggregate     684
Name: count, dtype: int64

In [22]:
df.loc[
    df["entity_type"] == "Aggregate",
    ["country", "country_code"]
].drop_duplicates().sort_values("country_code")

,country,country_code
8007,East Asia & Pacific (excluding high income),EAP
8008,Europe & Central Asia (excluding high income),ECA
8349,High income,HIC
8195,Latin America & Caribbean (excluding high income),LAC
8350,Low income,LIC
8351,Lower middle income,LMC
7950,Developing economies,LMY
8009,Middle East & North Africa (excluding high inc...,MNA
8196,South Asia,SAS
8010,Sub-Saharan Africa (excluding high income),SSA


## Mark Survey Period Type

The dataset mainly follows standard Global Findex survey waves. The 2022 observation covers only 16 countries, so they are retained but marked separately.

In [23]:
df["survey_period_type"] = "Standard survey wave"

df.loc[
    df["year"] == 2022,
    "survey_period_type",
] = "Off-cycle survey"

In [24]:
pd.crosstab(
    df["year"],
    df["survey_period_type"]
)

survey_period_type,Off-cycle survey,Standard survey wave
year,,
2011,0,1516
2014,0,1686
2017,0,1734
2021,0,1483
2022,176,0
2024,0,1982


## Standarize Text Values

Leading and trailing spaces are removed form text fields. Original capitalization and category wording are preserved.

In [25]:
text_columns = df.select_dtypes(include="object").columns

for column in text_columns:
    df[column] = df[column].str.strip()

In [26]:
empty_strings_counts = (df[text_columns] == "").sum()

empty_strings_counts[empty_strings_counts > 0]

Series([], dtype: int64)

## Final Misssing-Value Decisions

- Missing region, income-group, and adult-population values belong to aggregrate benchmark rows are retained.
- Missnig indicator values reflect differences in indicator avaiability across survey years and population segments.
- Missing indicator are preserved as null values because replacing them with zero would incorrectly mean that no people used the financial service
- No rows are removed based only on missing indicator values.

In [27]:
final_columns = [
    "country",
    "country_code",
    "entity_type",
    "year",
    "survey_period_type",
    "adult_population",
    "region",
    "income_group",
    "group",
    "group2",
    "account_ownership",
    "financial_institution_account",
    "mobile_money_account",
    "digital_access",
    "digital_merchant_payment",
    "digital_payment_made",
    "digital_payment_received",
    "inactive_account"
]

clean_df = df[final_columns].copy()

clean_df.head()

,country,country_code,entity_type,year,survey_period_type,adult_population,region,income_group,group,group2,account_ownership,financial_institution_account,mobile_money_account,digital_access,digital_merchant_payment,digital_payment_made,digital_payment_received,inactive_account
0,Afghanistan,AFG,Country,2011,Standard survey wave,14575546.0,South Asia (excluding high income),Low income,all,all,0.090050,0.090050,NaN,NaN,NaN,NaN,NaN,NaN
1,Albania,ALB,Country,2011,Standard survey wave,2281010.0,Europe & Central Asia (excluding high income),Upper middle income,all,all,0.282681,0.282681,NaN,NaN,NaN,NaN,NaN,NaN
2,Algeria,DZA,Country,2011,Standard survey wave,26251587.0,Middle East & North Africa (excluding high inc...,Lower middle income,all,all,0.332861,0.332861,NaN,NaN,NaN,NaN,NaN,NaN
3,Angola,AGO,Country,2011,Standard survey wave,12779501.0,Sub-Saharan Africa (excluding high income),Lower middle income,all,all,0.392035,0.392035,NaN,NaN,NaN,NaN,NaN,NaN
4,Argentina,ARG,Country,2011,Standard survey wave,30685516.0,Latin America & Caribbean (excluding high income),Upper middle income,all,all,0.331302,0.331302,NaN,NaN,NaN,NaN,NaN,NaN


In [28]:
clean_df.shape

(8577, 18)

## Final Dataset Validation

Validate the processed table before exporting it to SQL or Power BI

In [29]:
key_columns = ["country_code", "year", "group", "group2"]

invalid_indicator_values = (
    (clean_df[indicator_columns_clean] < 0) |
    (clean_df[indicator_columns_clean] > 1)
).sum().sum()

validation_results = pd.Series({
    "row_count": len(clean_df),
    "column_count": clean_df.shape[1],
    "missing_key_values": clean_df[key_columns].isna().sum().sum(),
    "duplicate_keys": clean_df.duplicated(subset=key_columns).sum(),
    "invalid_indicator_values": invalid_indicator_values,
    "country_rows": (clean_df["entity_type"] == "Country").sum(),
    "aggregate_rows": (clean_df["entity_type"] == "Aggregate").sum(),
    "off_cycle_rows": (clean_df["survey_period_type"] == "Off-cycle survey").sum()
})

validation_results

row_count                   8577
column_count                  18
missing_key_values             0
duplicate_keys                 0
invalid_indicator_values       0
country_rows                7893
aggregate_rows               684
off_cycle_rows               176
dtype: int64

## Export Analysis-Ready Dataset

Save the validated dataset for SQL analysis and Power BI. The processed file remains excluded from GitHub because it can be reproduced from this notebook.

In [30]:
PROCESSED_PATH = Path(
    r"D:/financial-inclusion-gap-analysis/data/processed/findex_analysis_ready.csv"
)

PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)

clean_df.to_csv(
    PROCESSED_PATH,
    index=False
)

In [31]:
PROCESSED_PATH.exists(), PROCESSED_PATH.stat().st_size

(True, 1737973)

In [32]:
processed_check = pd.read_csv(PROCESSED_PATH)

processed_check.shape

(8577, 18)

## Final Cleaning Summary

The cleaning process produced an analysis-ready dataset containing 8,577 rows and 18 columns.

### Validation Results
- The composite key contains no missing values.
- NO duplicate composite keys were found.
- All non-missing indiactors fail between 0 and 1.
- Country and aggregate benchmark rows are explicitly separated.
- The 2022 observation are retained and labelled as off-cycle survey rows.
- Missing indicators remain null because zero would represent a real measured outcome.
- NO rows were removed during cleaning.

### Output
The processed dataset was saved to `data/processed/findex_analysis_ready.csv`.

The file is ready for SQL analysis and Power BI.